# Seq2Seq walkthrough — Sutskever et al. 2014

Runnable tour of [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215)
([arXiv HTML](https://arxiv.org/html/1409.3215v3)). Implementation: `src/seq2seq/`.
Static notes: [`docs/architecture.md`](../docs/architecture.md), [`docs/configs.md`](../docs/configs.md).

**Not this paper:** attention (Bahdanau 2015).

### Paper vs this notebook

Same *structure*; this notebook uses a tiny **`toy`** budget so cells finish on CPU.

| | Paper (§2–§3.4) | This notebook |
|--|-----------------|---------------|
| L × H / embed | 4 × 1000 / 1000 | 2 × 64 / 64 |
| Vocab | 160k / 80k | synthetic |
| Data | WMT’14 En→Fr | synthetic strings |
| Batch / steps | 128 / 7.5 ep | 16 / ~20 steps |
| Goal | BLEU | shapes + loss moves |

**Still paper-faithful here:** reverse source, separate encoder/decoder LSTMs, Graves cell,
SGD + clip 5, init ±0.08, no attention.

```
 x_rev ──► Encoder LSTM×L ──► v=(h_T,c_T) ──► Decoder LSTM×L ──► softmax(|V_tgt|)
                                      ▲
                               y_<t> (teacher / decode)
```

$$p(y_1,\ldots,y_{T'}\mid x)=\prod_t p(y_t\mid v,\,y_{<t})$$


## 0. Setup

```bash
pip install -e ".[dev]"
# GPU image with CUDA torch already installed:
#   pip install -e ".[dev]" --no-deps && pip install datasets numpy tqdm pytest
```


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from seq2seq.config import toy_config, mid_config, paper_config
from seq2seq.data import make_synthetic_loader, basic_tokenize
from seq2seq.lstm_cell import LSTMCell
from seq2seq.model import Seq2Seq
from seq2seq.train import train
from seq2seq.decode import greedy_decode, beam_search_decode, encode_source_sentence

import torch

print("torch", torch.__version__, "cuda" if torch.cuda.is_available() else "cpu")
for name, fn in [("paper", paper_config), ("mid", mid_config), ("toy", toy_config)]:
    c = fn()
    print(f"{name:5} L={c.num_layers} H={c.hidden_size} V={c.src_vocab_size}/{c.tgt_vocab_size} B={c.batch_size}")


## 1. Source reversal (§2, §3.3)

Encoder sees tokens **right→left**; target stays left→right. Shortens the lag between
aligned early words (paper: large long-sentence BLEU gain).

```
EN:  the cat sat on the mat
ENC: mat the on sat cat the  →  v
DEC: <sos> … FR … <eos>
```


In [ ]:
src = "the cat sat on the mat"
toks = basic_tokenize(src)
print("tokens:", toks)
print("encoder order:", list(reversed(toks)))

cfg = toy_config()
cfg.max_steps = 20
cfg.log_every = 5
cfg.sample_every = 10
cfg.hidden_size = 64
cfg.embed_size = 64
cfg.num_layers = 2
cfg.batch_size = 16
cfg.checkpoint_dir = str(ROOT / "runs" / "notebook_toy")

loader, src_vocab, tgt_vocab, examples = make_synthetic_loader(cfg, n=128)
batch = next(iter(loader))
{k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in batch.items()}


## 2. Graves LSTM cell

Paper cites Graves (2013). `LSTMCell` implements gates explicitly (layout `[i|f|g|o]`):

$$
\begin{aligned}
i_t &= \sigma(W_{xi}x_t + W_{hi}h_{t-1} + b_i) &
f_t &= \sigma(W_{xf}x_t + W_{hf}h_{t-1} + b_f) \\
g_t &= \tanh(W_{xg}x_t + W_{hg}h_{t-1} + b_g) &
o_t &= \sigma(W_{xo}x_t + W_{ho}h_{t-1} + b_o) \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t &
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

Deep stack = $L$ cells; paper $L=4$, here $L=2$.


In [ ]:
cell = LSTMCell(input_size=8, hidden_size=16)
x = torch.randn(4, 8)
h = torch.zeros(4, 16)
c = torch.zeros(4, 16)
h2, c2 = cell(x, (h, c))
print("h", tuple(h2.shape), "c", tuple(c2.shape))


## 3. Encoder → $v$ → decoder (§2)

Decoder = LM conditioned on fixed $v$ (encoder final state). Softmax over full $|V_{\mathrm{tgt}}|$.

**Repo convention:** copy encoder final `(h,c)` into **all** decoder layers.

$$
p(y_1,\ldots,y_{T'}\mid x) = \prod_t p(y_t \mid v, y_{<t})
$$


In [ ]:
model = Seq2Seq.from_config(cfg, len(src_vocab), len(tgt_vocab), tgt_vocab.pad_id)
model.init_weights(cfg.init_range)
loss, logits = model(batch["src"], batch["src_lengths"], batch["tgt_in"], batch["tgt_out"])
print("loss", float(loss.detach()))
print("logits", tuple(logits.shape), "= (B, T_tgt, V)")
print("monitor groups:", list(model.param_groups_for_monitor()))


## 4. Train (§3.4 recipe, toy scale)

Paper: SGD LR $0.7$, clip if $\|g\|_2 > 5$, init $\pm 0.08$, length buckets.
Here: synthetic + `max_steps=20`.

`TrainingMonitor`: loss, ppl, LR, grad norms pre/post clip, $\|\Delta\theta\|$ per group.


In [ ]:
monitor = train(cfg, synthetic=True, sample_src="w0 w1 w2")
print("steps logged:", len(monitor.history))
if monitor.history:
    print(monitor.history[-1].summary())


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed; skip plot")

if plt and monitor.history:
    steps = [r.step for r in monitor.history]
    losses = [r.loss for r in monitor.history]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(steps, losses, marker="o")
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.set_title("Toy training loss")
    plt.show()

    last = monitor.history[-1]
    if last.delta_by_group:
        top = sorted(last.delta_by_group.items(), key=lambda kv: -kv[1])[:5]
        print("largest ‖Δθ‖ groups:", top)


## 5. Decode — greedy vs beam (§3.2)

Left-to-right beam search; paper: beam size **2** recovers most of the gain vs greedy.


In [ ]:
ckpt = torch.load(Path(cfg.checkpoint_dir) / "checkpoint.pt", weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()

src_t, lens = encode_source_sentence("w0 w1 w2", src_vocab, reverse=True)
g = greedy_decode(model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, max_len=20)[0]
b = beam_search_decode(
    model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, max_len=20, beam_size=2
)[0]
print("greedy:", tgt_vocab.decode(g))
print("beam2:", tgt_vocab.decode(b))


## 6. Next: real WMT on GPU

This notebook stays **toy/synthetic**. For **`mid`** (WMT'14, 4×256, load checkpoint, plot history, decode):

→ **[`02_mid_walkthrough.ipynb`](02_mid_walkthrough.ipynb)** on a CUDA pod, after:

```bash
python -m seq2seq.train --config mid --device cuda
```

### References

1. Sutskever, Vinyals, Le. *Sequence to Sequence Learning with Neural Networks*. NeurIPS 2014. [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)
2. Graves. *Generating Sequences With Recurrent Neural Networks*. 2013. [arXiv:1308.0850](https://arxiv.org/abs/1308.0850) (LSTM formulation cited by the paper)
3. Bahdanau, Cho, Bengio. *Neural Machine Translation by Jointly Learning to Align and Translate*. 2015 — **attention; not implemented here**
